# tp-05h アンサンブル v2（新しい特徴量のモデルを加えて再合成）＋ 弱いセグメントの再較正

**実行場所: ホスト PC（stg）。** tp-05g で採用した特徴量（追い切り評価・厩舎コメント・血統×条件）で学習した OOF（`lgb_extras*`・`xgb_extras_ped`・`nn_extras_ped`）を、多様性の源になる 3 モデル（lgb_softmax・linear・nn_s）と合わせて合成し直す（tp-05e の 9 モデル全部を再学習せずに済むよう候補を絞る。13 モデル版との差は `ensemble_v2_full13/` に参考で残す）。
評価は tp-05e と同じ leave-one-year-out（重みの過剰適合を含む正直な値）。**v1（tp-05e）との差**を、レース単位の paired bootstrap で確かめる。採用は上限 < 0 かつ連対・複勝が悪化しないこと。

## 弱いセグメントの再較正
障害・2700m 以上・少頭数（〜8 頭）は LogLoss が悪い（データが少ない）。各セグメントで、アンサンブル確率の logit に対する 1 次の補正（Platt）を **他の年だけで当てはめて** 当該年に適用する（LOYO）。全体の LogLoss が有意に良くなるときだけ採用する。

- 出力: `ensemble_v2/`（`ensemble_spec.json`・`loyo_logloss.csv` ほか）、OOF は `oof/model_oof/ensemble_v2.parquet`（v1 の `ensemble.parquet` は上書きしない）。
- 2026 年と test 2025 は使わない。

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.project_env import load_project_dotenv
load_project_dotenv()

In [ ]:
import json, subprocess
import numpy as np
import pandas as pd
import mlflow
from sklearn.linear_model import LogisticRegression

from src.pipeline.mlflow.train_utils import require_tracking_server
from src.pipeline.models import ensemble as en, tp05_data, win_models as wm, zoo_runner

WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
OOF_DIR = WORK / "oof/model_oof"
OUT = WORK / "ensemble_v2"
OUT.mkdir(parents=True, exist_ok=True)
mlflow.set_experiment("keiba-tp-ensemble")
require_tracking_server()
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()

D = tp05_data.load_dataset(WORK)
df, meta = D["df"], D["meta"]
NEW = ["lgb_extras_ped", "xgb_extras_ped", "nn_extras_ped", "lgb_extras"]
# 最終の再学習（tp-06b）の手間を抑えるため、新特徴量のモデルと、多様性の源になる 3 モデルに絞る（全 13 モデル版は ensemble_v2_full13/ に参考で残してある）
PREFER = NEW + ["lgb_softmax", "linear", "nn_s"]
names = [n for n in PREFER if (OOF_DIR / f"{n}.parquet").exists()]
print("合成する候補:", names)
key = ["race_id", "horse_id"]
base = meta[key + ["finish_position", "year"]].merge(pd.read_parquet(OOF_DIR / f"{names[0]}.parquet")[key], on=key, how="inner")
for n in names:
    o = pd.read_parquet(OOF_DIR / f"{n}.parquet")[key + ["p_win", "p_top2", "p_top3"]].rename(columns={f"p_{t}": f"{n}__{t}" for t in ("win", "top2", "top3")})
    base = base.merge(o, on=key, how="inner")
print(f"共通の行 {len(base):,}（{base['year'].min()}〜{base['year'].max()}）")
race, years, fin = base["race_id"].to_numpy(), base["year"].to_numpy(), base["finish_position"].to_numpy()
TOT = {"win": 1.0, "top2": 2.0, "top3": 3.0}
Y = {"win": (fin == 1).astype(float), "top2": (fin <= 2).astype(float), "top3": (fin <= 3).astype(float)}
P = {t: np.stack([base[f"{n}__{t}"].to_numpy() for n in names], axis=1) for t in TOT}

In [ ]:
rows, LOYO = [], {}
for t in TOT:
    for j, n in enumerate(names):
        rows.append({"target": t, "method": f"single:{n}", "logloss": en.ll(wm.normalize_to_total(race, P[t][:, j], TOT[t]), Y[t])})
    for m in en.METHODS:
        v, out = en.loyo_eval(m, P[t], years, race, Y[t], TOT[t])
        rows.append({"target": t, "method": m, "logloss": v})
        LOYO[(t, m)] = out
R = pd.DataFrame(rows)
piv = R.pivot(index="method", columns="target", values="logloss")[["win", "top2", "top3"]]
display(piv.sort_values("win").round(5))
piv.to_csv(OUT / "loyo_logloss.csv")
v1 = pd.read_csv(WORK / "ensemble/loyo_logloss.csv", index_col=0)
print("v1 の最良合成案:", v1.loc[[m for m in en.METHODS if m in v1.index]].min().round(5).to_dict())

In [ ]:
ORDER = ["mean", "logit_mean", "greedy", "weighted_logit", "stack_lr"]
CHOICE = {}
for t in TOT:
    sub = R[(R.target == t) & R.method.isin(ORDER)].set_index("method")["logloss"]
    CHOICE[t] = next(m for m in ORDER if sub[m] <= sub.min() + 5e-5)
print("採用する合成案:", CHOICE)
spec = {"git_commit": GIT_COMMIT, "models": names, "targets": {}}
final_P = {}
for t in TOT:
    m = CHOICE[t]
    Z = en.logit(P[t])
    st = en.fit_method(m, P[t], Z, Y[t])
    entry = {"method": m}
    if m in ("weighted_logit", "greedy"):
        entry["weights"] = [float(x) for x in st]
    elif m == "stack_lr":
        entry["coef"], entry["intercept"] = [float(x) for x in st.coef_[0]], float(st.intercept_[0])
    spec["targets"][t] = entry
    final_P[t] = LOYO[(t, m)]
    w = entry.get("weights")
    print(f"  {t}: {m}", {n: round(x, 3) for n, x in zip(names, w) if x > 0.005} if w else "")
(OUT / "ensemble_spec.json").write_text(json.dumps(spec, ensure_ascii=False, indent=1), encoding="utf-8")

In [ ]:
# v1 のアンサンブル（OOF）との比較: レース単位の paired bootstrap（負なら v2 が良い）
v1o = base[key].merge(pd.read_parquet(OOF_DIR / "ensemble.parquet")[key + ["p_win", "p_top2", "p_top3"]], on=key, how="left")
assert v1o["p_win"].notna().all()
rows = []
for t in TOT:
    p1 = v1o[f"p_{t}"].to_numpy()
    b = wm.paired_bootstrap_diff(wm.per_race_loss(race, p1, Y[t]), wm.per_race_loss(race, final_P[t], Y[t]), n_boot=1000)
    rows.append({"target": t, "v1": en.ll(p1, Y[t]), "v2": en.ll(final_P[t], Y[t]), "Δ(v2−v1)": b["diff"], "lo": b["lo"], "hi": b["hi"], "採用": b["hi"] < 0})
V2 = pd.DataFrame(rows).set_index("target")
display(V2.round(5)); V2.to_csv(OUT / "v2_vs_v1.csv")
ADOPT_V2 = bool(V2.loc["win", "採用"] and V2.loc["top2", "Δ(v2−v1)"] <= 2e-4 and V2.loc["top3", "Δ(v2−v1)"] <= 2e-4)
print("v2 を採用:", ADOPT_V2)

In [ ]:
# 弱いセグメントの再較正（LOYO）
seg_src = df[["race_id", "horse_id", "surface", "distance", "field_size"]].astype({"race_id": str, "horse_id": str})
sg = base[key].merge(seg_src, on=key, how="left")
jump = sg["surface"].astype(str).str.contains("障").to_numpy()
long_ = (sg["distance"].to_numpy() >= 2700) & ~jump
small = (sg["field_size"].to_numpy() <= 8) & ~jump & ~long_
SEG = np.where(jump, "障害", np.where(long_, "2700m以上(平地)", np.where(small, "〜8頭(平地)", "その他")))
print(pd.Series(SEG).value_counts().to_dict())
cal_P = {}
for t in TOT:
    z = en.logit(np.clip(final_P[t], 1e-6, 1 - 1e-6))
    out = final_P[t].copy()
    for s in ("障害", "2700m以上(平地)", "〜8頭(平地)"):
        for y in np.unique(years):
            tr_ = (SEG == s) & (years != y); te_ = (SEG == s) & (years == y)
            if tr_.sum() < 300 or te_.sum() == 0 or len(np.unique(Y[t][tr_])) < 2:
                continue
            lr = LogisticRegression(C=1e4, max_iter=200).fit(z[tr_].reshape(-1, 1), Y[t][tr_])
            out[te_] = lr.predict_proba(z[te_].reshape(-1, 1))[:, 1]
    cal_P[t] = wm.normalize_to_total(race, out, TOT[t])
rows = []
for t in TOT:
    b = wm.paired_bootstrap_diff(wm.per_race_loss(race, final_P[t], Y[t]), wm.per_race_loss(race, cal_P[t], Y[t]), n_boot=1000)
    rows.append({"target": t, "再較正前": en.ll(final_P[t], Y[t]), "再較正後": en.ll(cal_P[t], Y[t]), "Δ": b["diff"], "lo": b["lo"], "hi": b["hi"], "採用": b["hi"] < 0})
    for s in ("障害", "2700m以上(平地)", "〜8頭(平地)"):
        m_ = SEG == s
        rows[-1][f"Δ {s}"] = en.ll(cal_P[t][m_], Y[t][m_]) - en.ll(final_P[t][m_], Y[t][m_])
CAL = pd.DataFrame(rows).set_index("target")
display(CAL.round(5)); CAL.to_csv(OUT / "segment_recalibration.csv")
ADOPT_CAL = bool(CAL.loc["win", "採用"] and CAL.loc["top2", "Δ"] <= 2e-4 and CAL.loc["top3", "Δ"] <= 2e-4)
print("セグメント再較正を採用:", ADOPT_CAL)

In [ ]:
oof = base[key + ["year"]].copy()
for t in TOT:
    oof[f"p_{t}"] = final_P[t]
oof.to_parquet(OOF_DIR / "ensemble_v2.parquet", index=False)
oof_c = oof.copy()
for t in TOT:
    oof_c[f"p_{t}"] = cal_P[t]
oof_c.to_parquet(OOF_DIR / "ensemble_v2_cal.parquet", index=False)
ens = zoo_runner.metrics_by_year(base[["year"]].reset_index(drop=True), final_P, fin, race, "ensemble_v2")
ens.to_csv(OUT / "ensemble_v2_fold_metrics.csv"); display(ens.round(5))
(OUT / "decision.json").write_text(json.dumps({"adopt_v2": ADOPT_V2, "adopt_segment_recalibration": ADOPT_CAL, "git_commit": GIT_COMMIT, "models": names, "choice": CHOICE}, ensure_ascii=False, indent=1))
with mlflow.start_run(run_name="tp-05h-ensemble-v2", tags={"notebook": "tp-05h", "git_commit": GIT_COMMIT, "models": ",".join(names)}):
    mlflow.log_params({f"method_{t}": CHOICE[t] for t in TOT} | {"adopt_v2": ADOPT_V2, "adopt_recal": ADOPT_CAL})
    mlflow.log_metrics({f"v2_{t}": float(V2.loc[t, "v2"]) for t in TOT} | {f"v1_{t}": float(V2.loc[t, "v1"]) for t in TOT} | {f"delta_{t}": float(V2.loc[t, "Δ(v2−v1)"]) for t in TOT})
    for f_ in ("ensemble_spec.json", "loyo_logloss.csv", "v2_vs_v1.csv", "segment_recalibration.csv", "decision.json"):
        mlflow.log_artifact(str(OUT / f_), "ensemble_v2")